# Validate normal synthetic banking behaviour
This notebook inspects the clean baseline before any fraud typologies are injected.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/raw')
customers = pd.read_parquet(DATA / 'customers.parquet')
accounts = pd.read_parquet(DATA / 'accounts.parquet')
transactions = pd.read_parquet(DATA / 'transactions.parquet')
{name: frame.shape for name, frame in {'customers': customers, 'accounts': accounts, 'transactions': transactions}.items()}

{'customers': (50000, 14),
 'accounts': (70000, 6),
 'transactions': (100000, 16)}

In [2]:
transactions['amount'].describe(percentiles=[.5, .9, .95, .99]).round(2)

count    100000.00
mean       1174.81
std        1588.72
min           0.22
50%         699.08
90%        2573.42
95%        3751.92
99%        7433.68
max       61497.87
Name: amount, dtype: float64

In [3]:
transactions['channel'].value_counts(normalize=True).mul(100).round(2)

channel
CARD      44.01
ONLINE    17.79
MOBILE    16.97
EFT       13.12
ATM        8.11
Name: proportion, dtype: float64

In [4]:
profile = transactions.merge(customers[['customer_id', 'avg_transaction']], on='customer_id')
profile['amount_vs_customer_average'] = profile['amount'] / profile['avg_transaction']
profile['amount_vs_customer_average'].describe(percentiles=[.5, .9, .95, .99]).round(2)

count    100000.00
mean          1.16
std           0.73
min           0.00
50%           1.00
90%           1.97
95%           2.46
99%           3.86
max          14.96
Name: amount_vs_customer_average, dtype: float64

In [5]:
assert accounts.customer_id.isin(customers.customer_id).all()
assert transactions.account_id.isin(accounts.account_id).all()
assert transactions.customer_id.isin(customers.customer_id).all()
assert transactions[['balance_before', 'balance_after']].ge(0).all().all()
assert {'is_fraud', 'fraud_rule', 'fraud_scenario'}.isdisjoint(transactions.columns)
print('All baseline checks passed.')

All baseline checks passed.
